<a id="top"></a>

# Demo: build a minimal MCP server (shown, not run here)

```yaml
title: "Demo: build a minimal MCP server (shown, not run here)"
keywords: mcp, server, stdio, book_meeting, tool spec, error shape, structured result, not-runnable, l05 design, teacher demo
estimated duration: 18
```

> **Lesson:** L09.
>
> **⚠️ NOT RUNNABLE in this environment.** The Python `mcp` package is **not installed** in the course env, so the server code cells below are **shown for their shape, not executed**. This notebook is committed **without running** the `mcp`-dependent cells (no execution counts, no outputs on them). Read them to see the server skeleton; the *runnable* offline counterpart is the [L0903 demo](L0903_lecture.ipynb).
>
> **To actually run it later:** `uv add mcp`, then run the server module on stdio and point an MCP client at it (see the [L0905 connect walkthrough](L0905_lecture.md)).
>
> **The point to land:** the same [L08](../L08/L0802_lecture.md) `book_meeting` tool becomes an MCP server with **no design change** — only the packaging (a registration call, a transport, a `__main__` entry point) is new. Error shaping must survive the boundary.

## Contents

- [1. The pure-Python core (this part is just L08)](#1-the-pure-python-core-this-part-is-just-l08)
- [2. Wrap it in an MCP server (needs `mcp`)](#2-wrap-it-in-an-mcp-server-needs-mcp)
- [3. Error shaping must survive the boundary](#3-error-shaping-must-survive-the-boundary)
- [4. What a client discovers (the published spec)](#4-what-a-client-discovers-the-published-spec)
- [5. Takeaways](#5-takeaways)

## 1. The pure-Python core (this part is just L08)

The tool's **implementation** is plain Python and has nothing to do with MCP. It is the [L08](../L08/L0802_lecture.md) `book_meeting` design: a tight check on each field, and an **informative structured error** (`{error, field, message}`) instead of a bare exception. This cell is ordinary Python — but we keep it un-run here so the whole notebook reads as a single not-executed walkthrough.

In [ ]:
# (Shown, not run here — part of the not-executed walkthrough.)
# The tool implementation. Note: ZERO mcp imports. This is the L08 design,
# unchanged. The error shape is the L08 {error, field, message} contract.
def book_meeting(attendee: str, start: str, duration_minutes: int = 30) -> dict[str, object]:
    """Book a meeting, or return a structured validation error the model can fix."""
    if "@" not in attendee:
        return {
            "error": "validation",
            "field": "attendee",
            "message": f"must be an email address, got {attendee!r}",
        }
    if not (15 <= duration_minutes <= 240):
        return {
            "error": "validation",
            "field": "duration_minutes",
            "message": f"must be an integer 15-240, got {duration_minutes}",
        }
    # The real implementation would write to a calendar here.
    return {
        "booked": True,
        "attendee": attendee,
        "start": start,
        "duration_minutes": duration_minutes,
    }

[↑ Back to top](#top)

## 2. Wrap it in an MCP server (needs `mcp`)

Now the **packaging**. We register the tool with an MCP server, declare its spec (name + description + `inputSchema` — the same surface the [L0903 demo](L0903_lecture.ipynb) translated), and choose **stdio** as the transport. This is the only genuinely new code, and it needs the `mcp` package — **not installed here, so this cell does not run.**

In [ ]:
# ⚠️ NOT-RUNNABLE here: requires `uv add mcp`. Shown for shape only.
from mcp.server.fastmcp import FastMCP  # type: ignore[import-not-found]

server = FastMCP("book-meeting-server")


@server.tool()
def book_meeting_tool(attendee: str, start: str, duration_minutes: int = 30) -> dict[str, object]:
    """Book a meeting on the user's calendar. Use this when the user asks to
    schedule, set up, or book a meeting with a named attendee at a given time.

    The docstring + type hints become the published tool spec the client
    discovers (name, description, inputSchema) — the L08 design surface, on the wire."""
    # Delegate to the pure-Python core from section 1. The structured error it
    # returns becomes the tool RESULT — it does NOT crash the server.
    return book_meeting(attendee, start, duration_minutes)


if __name__ == "__main__":
    # Run on stdio: the client launches this module as a child process and pipes to it.
    server.run(transport="stdio")

[↑ Back to top](#top)

## 3. Error shaping must survive the boundary

In-process, an uncaught exception is a stack trace you can catch. **Across the boundary, an uncaught exception can crash the server** — taking the tool offline mid-conversation. The discipline (lecture slide 4.3): **catch it inside the tool and return a structured result**, so the model gets actionable context *and* the server stays up. Our section-1 core already does this — it returns `{error, field, message}` instead of raising. The table shows the two paths.

| A `duration_minutes: 999999` call | What the model sees | Server health |
| --- | --- | --- |
| exception escapes the tool | a transport-level error (opaque) | **server may crash** |
| tool returns `{error, field, message}` | actionable validation error, fixes its call | **server stays up** |

This is [L08](../L08/L0802_lecture.md)'s error design, now load-bearing for *availability*, not just helpfulness.

[↑ Back to top](#top)

## 4. What a client discovers (the published spec)

When a client connects (see [L0905](L0905_lecture.md)) and asks for the tool list, the server publishes this spec — derived from the function's name, docstring, and type hints. It is the **same** `book_meeting` spec the [L0903 demo](L0903_lecture.ipynb) produced by translating the inline definition. Design unchanged; only the path to the model is new.

```json
{
  "name": "book_meeting_tool",
  "description": "Book a meeting on the user's calendar. Use this when the user asks to schedule, set up, or book a meeting...",
  "inputSchema": {
    "type": "object",
    "properties": {
      "attendee": {"type": "string"},
      "start": {"type": "string"},
      "duration_minutes": {"type": "integer", "default": 30}
    },
    "required": ["attendee", "start"]
  }
}
```

[↑ Back to top](#top)

## 5. Takeaways

- An MCP server is a **small program**, not a microservice: a tool function + a registration call + a transport + a `__main__` block. Start small (lecture slide 4.1).
- The [L08](../L08/L0802_lecture.md) design — name, description, schema, **error shape** — is **unchanged**. The packaging is the only new code, and it's the part that needed `mcp`.
- **Catch exceptions and return structured results**, or an uncaught error becomes a transport-level crash. Error design now protects *availability*.
- The **server author owns the published tool list** — it's runtime system prompt for every connecting model. Curate it (back-reference [L08](../L08/L0802_lecture.md): *more tools ≠ more capable agent*).
- Next: decide *whether* to pay this packaging tax at all — the [L0907 lab](L0907_lab_empty.ipynb) turns the cost/benefit ledger into a decision function.

[↑ Back to top](#top)